
# DUT Anti-UAV Detection — YOLOv8n Baseline

## Project Overview

This notebook implements and evaluates a **YOLOv8 Nano (YOLOv8n)** object detection baseline using the **DUT Anti-UAV dataset**.

The objective is to fine-tune a pretrained YOLOv8n model for UAV detection, assess its performance on the official dataset splits, and establish a baseline for future model comparisons and tracking integration.

### Experimental Pipeline

1. Configure the Google Colab GPU environment.
2. Load and prepare the DUT Anti-UAV dataset.
3. Configure YOLO dataset paths and classes.
4. Fine-tune YOLOv8n using pretrained weights.
5. Evaluate the best checkpoint on the official test set.
6. Analyze detection performance and identify limitations.

**Dataset:** DUT Anti-UAV (RGB detection subset)  
**Framework:** Ultralytics YOLOv8  
**Hardware:** NVIDIA Tesla T4 GPU  
**Task:** Single-class object detection (UAV)



## 1. Environment Setup

This section prepares the Google Colab environment for model training and evaluation.

The following operations are performed:

- Mount Google Drive to access datasets, trained models, and experiment results.
- Verify NVIDIA GPU availability.
- Install the required Ultralytics version.
- Confirm that PyTorch can access CUDA.

**Note:** Google Colab runtimes are temporary. These setup steps may need to be repeated when starting a new session.


In [ ]:
from google.colab import drive
from pathlib import Path

# Connect Google Drive
drive.mount("/content/drive")

# Our project storage
BASE = Path("/content/drive/MyDrive/JuniorProject")

# Trained model and dataset archive
checkpoint_path = BASE / "Training_Results/yolov8n_dut_baseline/weights/best.pt"
dataset_zip = BASE / "dut_yolo_dataset.zip"

print("Best model exists:", checkpoint_path.exists())
print("Dataset ZIP exists:", dataset_zip.exists())

if checkpoint_path.exists():
    print("Model size:", round(checkpoint_path.stat().st_size / 1024**2, 2), "MB")

Mounted at /content/drive
Best model exists: True
Dataset ZIP exists: True
Model size: 6.2 MB


In [ ]:
!nvidia-smi

Mon Oct 26 18:38:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%pip install -q ultralytics==8.4.175

In [ ]:
import torch
import ultralytics

print("Ultralytics:", ultralytics.__version__)
print("CUDA available:", torch.cuda.is_available())

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics: 8.4.175
CUDA available: True



## 2. Dataset Preparation

The DUT Anti-UAV dataset contains 10,000 detection images, divided into the official training, validation, and testing splits.

| Dataset Split | Images |
|---|---:|
| Training | 5,200 |
| Validation | 2,600 |
| Testing | 2,200 |
| **Total** | **10,000** |

The original Pascal VOC XML annotations were previously converted into YOLO-compatible normalized bounding-box annotations.

The converted dataset is stored as a ZIP archive in Google Drive. It is copied and extracted into Colab's local `/content/` directory to improve data-loading performance.

The dataset structure is:

```text
dataset/
├── images/
│   ├── train/
│   ├── val/
│   └── test/
└── labels/
    ├── train/
    ├── val/
    └── test/
```


In [ ]:
from pathlib import Path
import shutil
import zipfile

# Dataset stored permanently in Google Drive
zip_drive = Path(
    "/content/drive/MyDrive/JuniorProject/dut_yolo_dataset.zip"
)

# Temporary Colab copy
zip_local = Path("/content/dut_yolo_dataset.zip")

# Copy ZIP from Google Drive
shutil.copy2(zip_drive, zip_local)

# Extract into Colab's local storage
with zipfile.ZipFile(zip_local, "r") as zip_ref:
    zip_ref.extractall("/content")

print("Dataset extracted successfully!")

Dataset extracted successfully!


In [ ]:
from pathlib import Path

DATASET = Path("/content/dataset")

for split in ["train", "val", "test"]:
    images = list((DATASET / "images" / split).glob("*.jpg"))
    print(f"{split}: {len(images)} images")

train: 5200 images
val: 2600 images
test: 2200 images



## 3. YOLO Dataset Configuration

The `data.yaml` file defines the dataset location, training/validation/testing splits, and target classes.

For the current baseline, only one class is considered:

**Class 0: UAV**

The original DUT dataset splits are preserved to support reproducible evaluation and comparisons with published research.

All model training and evaluation use the YOLO-formatted annotations generated during local dataset preprocessing.


In [ ]:
%%writefile /content/data.yaml
path: /content/dataset

train: images/train
val: images/val
test: images/test

names:
  0: UAV

Writing /content/data.yaml



## 4. YOLOv8n Baseline Training

A pretrained YOLOv8n model is fine-tuned on the DUT Anti-UAV training set.

### Training Configuration

| Parameter | Value |
|---|---|
| Model | YOLOv8n |
| Initial weights | `yolov8n.pt` |
| Epochs | 50 |
| Input image size | 640 × 640 |
| Batch size | 16 |
| GPU | NVIDIA Tesla T4 |
| Training images | 5,200 |
| Validation images | 2,600 |
| Random seed | 0 |
| Framework | Ultralytics 8.4.175 |

### Training Outputs

The training process produces:

- `best.pt` — checkpoint selected using validation performance.
- `last.pt` — checkpoint from the final epoch.
- Training and validation loss curves.
- Evaluation metrics, plots, and experiment configuration.

**Training status:** Completed successfully (50 epochs).

**Important:** The training cell is preserved for reproducibility and should not be rerun during test-only evaluation. Use the existing `best.pt` checkpoint.


In [ ]:
RUN_TRAINING = False

from ultralytics import YOLO

# Load pretrained YOLOv8 nano weights
model = YOLO("yolov8n.pt")

if RUN_TRAINING:
  # Train the baseline model
  results = model.train(
      data="/content/data.yaml",
      epochs=50,
      imgsz=640,
      batch=16,
      device=0,
      workers=2,
      project="/content/drive/MyDrive/JuniorProject/Training_Results",
      name="yolov8n_dut_baseline",
      seed=0,
      deterministic=True
  )
else:
    print("Skipping training. Using the existing checkpoint.")

Ultralytics 8.4.175 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_dut_baseline, nbs=64, nms=None, opset=None, optimize=False, op


## 5. Official Test-Set Evaluation

The best-performing checkpoint from the training process is evaluated on the official DUT Anti-UAV test set.

The test set contains **2,200 images and 2,245 annotated UAV instances**.

### Evaluation Procedure

1. Load the fine-tuned `best.pt` checkpoint.
2. Run detection on the official test split.
3. Compare predicted bounding boxes against ground-truth annotations.
4. Calculate the detection evaluation metrics.

### Evaluation Metrics

- **Precision:** Proportion of positive detections that are correct.
- **Recall:** Proportion of ground-truth UAVs successfully detected.
- **mAP@50:** Mean Average Precision at an IoU threshold of 0.50.
- **mAP@50–95:** Mean Average Precision averaged over IoU thresholds from 0.50 to 0.95.

The test split was not used for model training or checkpoint selection.


In [ ]:
from ultralytics import YOLO

# Location of the trained model
MODEL_PATH = (
    "/content/drive/MyDrive/JuniorProject/"
    "Training_Results/yolov8n_dut_baseline/weights/best.pt"
)

# Load our fine-tuned model, not the original pretrained model
checkpoint_path = YOLO(MODEL_PATH)

# Evaluate on the official DUT test set
test_metrics = checkpoint_path.val(
    data="/content/data.yaml",
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    plots=True,
    project="/content/drive/MyDrive/JuniorProject/Training_Results",
    name="yolov8n_dut_test"
)

Ultralytics 8.4.175 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,132,707 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2617.9±1021.2 MB/s, size: 181.1 KB)
val: Scanning /content/dataset/labels/test... 2200 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2200/2200 1.9Kit/s 1.2s
val: New cache created: /content/dataset/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 138/138 5.5it/s 24.9s
                   all       2200       2245      0.954      0.891       0.93      0.609
Speed: 0.9ms preprocess, 2.8ms inference, 0.0ms loss, 1.7ms postprocess per image
Results saved to /content/drive/MyDrive/JuniorProject/Training_Results/yolov8n_dut_test


In [ ]:
print("DUT Anti-UAV — YOLOv8n Test Results")
print("-----------------------------------")

print(f"Precision:  {test_metrics.box.mp:.4f}")
print(f"Recall:     {test_metrics.box.mr:.4f}")
print(f"mAP@50:     {test_metrics.box.map50:.4f}")
print(f"mAP@50-95:  {test_metrics.box.map:.4f}")

DUT Anti-UAV — YOLOv8n Test Results
-----------------------------------
Precision:  0.9544
Recall:     0.8913
mAP@50:     0.9301
mAP@50-95:  0.6090



## 6. Experimental Results

### Quantitative Performance

The YOLOv8n baseline was evaluated using the official DUT Anti-UAV validation and test splits.

| Metric | Validation | Test |
|---|---:|---:|
| Precision | 0.9360 | 0.9544 |
| Recall | 0.8270 | 0.8913 |
| mAP@50 | 0.8700 | 0.9301 |
| mAP@50–95 | 0.5370 | 0.6090 |

### Preliminary Observations

- The model achieves strong UAV detection precision on both splits.
- Test-set recall and mAP are higher than the corresponding validation results.
- Performance at stricter IoU thresholds is lower than mAP@50, indicating room for improvement in bounding-box localization.
- Further qualitative analysis is needed to investigate missed detections, false positives, and small/distant UAVs.

**Note:** These results establish our first YOLOv8n baseline. Direct comparisons with published methods require consistent evaluation protocols.



## 7. Future Work

The following tasks will extend the initial UAV detection baseline:

1. **Qualitative Evaluation:** Visualize model predictions on test images and inspect bounding-box quality.
2. **Failure Analysis:** Identify false positives, false negatives, and difficult detection scenarios, particularly small and distant UAVs.
3. **Benchmark Comparison:** Compare the YOLOv8n baseline with detection methods evaluated in the original DUT Anti-UAV paper and relevant recent studies.
4. **Model Improvement:** Investigate alternative input resolutions, training configurations, or detection architectures based on evaluation findings.
5. **Object Tracking:** Integrate an appropriate tracking algorithm, such as ByteTrack, after finalizing the detection baseline.
6. **Reproducibility:** Maintain experiment configurations, source code, evaluation results, and documentation through GitHub and Google Drive.

### Research Objective

Establish a reliable UAV detection and tracking pipeline that can later contribute to the broader multi-agent drone swarm system.
